# CFM Pipeline Test
Load CFM model, pass an image, get concept vector. Verify pipeline works before smoothing experiments.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
from PIL import Image
from torchvision import transforms
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.5.1+cu121
CUDA available: False


In [ ]:
# Load CFM Model (CLIP-DINOiser + SAE)
from cfm.arg_parser import get_default_parser
from cfm.utils import common_init, get_img_model
from cfm.cfm import CFM
from dictionary_learning.utils import load_dictionary

# SAE config name (from Kai's directory)
CONFIG_NAME = 'k_12_ef_16_lr_0.0001_mf_[0.008,0.03,0.06,0.12,0.24,0.542]'

# Build args from config.py defaults
parser = get_default_parser()
args = parser.parse_args([])
args.device = 'cuda' if torch.cuda.is_available() else 'cpu'
common_init(args)
args.config_name = CONFIG_NAME

print("Device:", args.device)
print("Image encoder:", args.img_enc_name)
print("SAE checkpoint dir:", args.save_dir_sae_ckpts['img'])

c:\Daten\D_Part\Personal\CFM_smoothing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# Load Feature Extractor (CLIP-DINOiser)
feature_extractor, preprocess = get_img_model(args)
feature_extractor.eval()
print("CLIP-DINOiser loaded")

In [ ]:
# Load SAE 
sae_base = Path(args.save_dir_sae_ckpts['img']) / args.config_name / 'trainer_0'
print("Looking for SAE at:", sae_base)
print("Exists:", sae_base.exists())

if sae_base.exists():
    autoencoder, ae_config = load_dictionary(str(sae_base), args.device)
    print("SAE loaded")
    print("SAE config:", ae_config)
else:
    # List what's available
    sae_parent = Path(args.save_dir_sae_ckpts['img'])
    print("Available SAE configs:")
    if sae_parent.exists():
        for d in sae_parent.iterdir():
            print(f"  - {d.name}")
    else:
        print(f"  SAE parent dir doesn't exist: {sae_parent}")
        print("  Check config.py paths!")

In [ ]:
# Create CFM model
cfm_model = CFM(
    feature_extractor=feature_extractor,
    autoencoder=autoencoder,
    apply_found=False,
    device=args.device,
)
cfm_model.eval()
print("CFM model ready")

In [ ]:
# Load concept names
concept_name_save_path = os.path.join(
    args.save_dir_sae_ckpts['img'], 
    args.save_suffix, 
    args.config_name, 
    'trainer_0', 
    'concept_names.txt'
)
print("Looking for concept names at:", concept_name_save_path)

if os.path.exists(concept_name_save_path):
    with open(concept_name_save_path, "r") as f:
        concept_names = [line.strip() for line in f.readlines()]
    print(f"Loaded {len(concept_names)} concept names")
    print("First 20:", concept_names[:20])
else:
    concept_names = None
    print("Concept names file not found, using indices instead.")

## Test image through pipeline

In [ ]:
# Load a test image
import urllib.request
test_img_path = "test_image.jpg"
if not os.path.exists(test_img_path):
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
    urllib.request.urlretrieve(url, test_img_path)
    print("Downloaded test image")

# Option B: Use an image from a dataset
# Load and preprocess
image = Image.open(test_img_path).convert("RGB")
image_tensor = preprocess(image).unsqueeze(0).to(args.device)

print(f"Image size: {image.size}")
print(f"Tensor shape: {image_tensor.shape}")
print(f"Image size: {image.size}")
# Display
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Test Image")
plt.axis("off")
plt.show()

In [ ]:
# Get concept vector
with torch.no_grad():
    concept_vector = cfm_model.get_aggregated_concept_activations(image_tensor)  # [1, 8192]
    concept_maps = cfm_model.get_concept_activation_map(image_tensor)  # [1, 8192, 28, 28]

print("=" * 60)
print("CONCEPT VECTOR (image-level, max-pooled)")
print("=" * 60)
print(f"Shape: {concept_vector.shape}")
print(f"Non-zero concepts: {(concept_vector[0] > 0).sum().item()} / {concept_vector.shape[1]}")
print(f"Min (non-zero): {concept_vector[concept_vector > 0].min().item():.4f}")
print(f"Max: {concept_vector.max().item():.4f}")
print(f"Mean (non-zero): {concept_vector[concept_vector > 0].mean().item():.4f}")

print(f"\n{'=' * 60}")
print("CONCEPT MAPS (spatial)")
print("=" * 60)
print(f"Shape: {concept_maps.shape}")

# Top active concepts
top_k = 20
top_values, top_indices = concept_vector[0].topk(top_k)
print(f"\nTop {top_k} active concepts:")
for i, (idx, val) in enumerate(zip(top_indices, top_values)):
    name = concept_names[idx.item()] if concept_names else f"concept_{idx.item()}"
    print(f"  {i+1:2d}. [{idx.item():4d}] {name:30s} = {val.item():.4f}")

In [ ]:
# Noise stability test
import torch.nn.functional as F

def get_top_concepts(cfm_model, image_tensor, top_k=20):
    """Get top-k concept indices and values for an image"""
    with torch.no_grad():
        cv = cfm_model.get_aggregated_concept_activations(image_tensor)
    values, indices = cv[0].topk(top_k)
    return set(indices.tolist()), cv

print("Quick noise stability test:")
print("=" * 60)

original_set, original_cv = get_top_concepts(cfm_model, image_tensor, top_k=20)

for sigma in [0.01, 0.05, 0.1, 0.2, 0.5]:
    noise = sigma * torch.randn_like(image_tensor)
    noisy_image = (image_tensor + noise).clamp(0, 1)
    
    noisy_set, noisy_cv = get_top_concepts(cfm_model, noisy_image, top_k=20)
    
    # Jaccard similarity
    intersection = original_set & noisy_set
    union = original_set | noisy_set
    jaccard = len(intersection) / len(union) if union else 1.0
    
    # Cosine similarity of full concept vectors
    cosine = F.cosine_similarity(original_cv, noisy_cv, dim=1).item()
    
    # Concepts that flipped
    lost = original_set - noisy_set
    gained = noisy_set - original_set
    
    print(f"\nσ = {sigma:.2f}:")
    print(f"  Jaccard(top-20): {jaccard:.3f} | Cosine: {cosine:.4f}")
    print(f"  Kept: {len(intersection)}/20 | Lost: {len(lost)} | Gained: {len(gained)}")
    if lost and concept_names:
        lost_names = [concept_names[i] for i in list(lost)[:5]]
        print(f"  Lost concepts: {lost_names}")
    if gained and concept_names:
        gained_names = [concept_names[i] for i in list(gained)[:5]]
        print(f"  Gained concepts: {gained_names}")

In [ ]:
# Visualize concept activation maps
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# Show original image
axes[0, 0].imshow(image)
axes[0, 0].set_title("Original Image")
axes[0, 0].axis("off")

# Show top 9 concept activation maps
top_values, top_indices = concept_vector[0].topk(9)
for i in range(9):
    ax = axes[(i + 1) // 5, (i + 1) % 5]
    idx = top_indices[i].item()
    name = concept_names[idx] if concept_names else f"concept_{idx}"
    
    # Get spatial activation for this concept
    activation = concept_maps[0, idx].cpu().numpy()
    
    ax.imshow(activation, cmap='hot', interpolation='bilinear')
    ax.set_title(f"{name}\n({top_values[i].item():.2f})", fontsize=9)
    ax.axis("off")

plt.suptitle("Top 9 Concept Activation Maps", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()